# ESR under unseen conditions — Kaggle runner

All code lives in the GitHub repo; this notebook only clones it and runs the scripts.

**Before running:** *Add Input* → `yousirui1/fsd50k`; *Settings* → **Accelerator: GPU T4**, **Internet: On**.
Run one experiment per committed version (*Save Version → Save & Run All*): `cnn_baseline`, `effnet_standard`, `effnet_robust`, `effnet_robust_v2`, or the ablations `effnet_mixstyle`, `effnet_corrupt`, `effnet_eq`.

## 1. Get the code

In [ ]:
REPO_URL = "https://github.com/<your-user>/<your-repo>.git"  # <- your repo
BRANCH = "main"
!git clone -b $BRANCH $REPO_URL /kaggle/working/esr
%cd /kaggle/working/esr
!git pull origin $BRANCH
!pip install -q "timm>=1.0"

## 2. Settings

In [ ]:
EXPERIMENT = "effnet_robust_v2"   # cnn_baseline | effnet_standard | effnet_robust | effnet_robust_v2 | effnet_mixstyle | effnet_corrupt | effnet_eq
DATA_PATH = "/kaggle/input/datasets/yousirui1/fsd50k/fsd50k"  # or omit --data-path to auto-discover
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2b. Run several experiments in one session (optional)

A Kaggle session gives up to 12 h; one experiment's train+evaluate takes about 2.5–3 h (1h45 train + 40–60 min evaluate for protocol v2). You can fit 2 experiments in one session. Skip this and use sections 3–4 instead if you only want the single `EXPERIMENT` set above.

Status so far (see `docs/EXPERIMENT.md`): `effnet_robust` is done (Experiment 2a). Pick the next session's list from the Kaggle Run Plan in `docs/superpowers/plans/2026-09-27-esr-v2-next-steps.md`:
- **Session 1 (baselines):** `["effnet_standard", "cnn_baseline"]`
- **Session 2 (v2):** `["effnet_robust_v2", "effnet_eq"]`
- **Session 3 (ablation):** `["effnet_mixstyle", "effnet_corrupt"]`

In [ ]:
EXPERIMENTS_THIS_SESSION = ["effnet_standard", "cnn_baseline"]  # edit per session, see the markdown above

for exp in EXPERIMENTS_THIS_SESSION:
    get_ipython().system(f"python scripts/train.py --experiment {exp} --data-path {DATA_PATH}")
    get_ipython().system(f"python scripts/evaluate.py --experiment {exp} --data-path {DATA_PATH}")

## 3. Smoke test (~15 min): 2,000 train clips, 1,000 eval clips, 2 epochs → `/kaggle/working/outputs_quick`

In [ ]:
!python scripts/train.py --experiment $EXPERIMENT --data-path $DATA_PATH --quick
!python scripts/evaluate.py --experiment $EXPERIMENT --data-path $DATA_PATH --quick --severities 3

## 4. Full run → `/kaggle/working/outputs/<experiment>/`
Resumes automatically from `last.pt`. After a 12 h timeout, add the previous version's output as an input and append
`--resume-from /kaggle/input/<that-output>/outputs/<experiment>/last.pt`.

In [ ]:
!python scripts/train.py --experiment $EXPERIMENT --data-path $DATA_PATH
!python scripts/evaluate.py --experiment $EXPERIMENT --data-path $DATA_PATH

## 4b. Re-evaluate a checkpoint from an earlier version (protocol v2)
Add that version's output as an input, then set `CKPT` to its `best.pt` path to enable this cell. Results go to `/kaggle/working/outputs/<experiment>/`.

In [ ]:
CKPT = ""  # e.g. "/kaggle/input/<earlier-version-output>/outputs/effnet_robust/best.pt"; leave "" to skip
if CKPT:  # opt-in: this re-evaluates all 22 conditions (~40-60 min GPU). Do not enable in a version that also retrains effnet_robust (same output folder).
    get_ipython().system(f"python scripts/evaluate.py --experiment effnet_robust --data-path {DATA_PATH} --checkpoint {CKPT}")

## 5. Compare experiments
Add the finished versions' outputs as inputs; this picks up every `robustness.csv` (smoke-test results are excluded).

In [ ]:
!python scripts/compare.py --out /kaggle/working/summary.csv

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
table = pd.read_csv("/kaggle/working/summary.csv")
display(table.round(4))
table.set_index("experiment")[["clean", "seen", "unseen"]].plot.bar(title="mAP: clean vs seen vs unseen conditions")
plt.ylabel("mAP"); plt.show()
detail = pd.read_csv("/kaggle/working/summary_by_condition.csv")
display(detail.pivot_table(index=["group", "condition", "severity"], columns="experiment", values="rel").round(3))